In [0]:
from pyspark.sql.functions import col, upper, trim, round, when
dados_brutos = spark.table("workspace.default.antidepressivos_brasil_unificado")

tabela_limpa = (
    dados_brutos
    .select(
        # Identificadores e Tempo
        col("id_linha").cast("int").alias("id_linha"),
        col("ano").cast("int").alias("ano_referencia"),
        
        # Tratamento de Texto (Remove espaços e padroniza em letras maiúsculas)
        upper(trim(col("regiao"))).alias("regiao_nome"),
        upper(trim(col("estado"))).alias("estado_nome"),
        
        # Correção de Valores Nulos na UF (Se estiver vazio, vira 'BR' Nacional)
        when(col("uf").isNull(), "BR")
        .otherwise(upper(col("uf")))
        .alias("uf_sigla"),
        
        # Ajuste de Métricas de Vendas (Arredonda e converte para número inteiro)
        round(col("unidades_vendidas_est"), 0).cast("long").alias("vendas_unidades"),
        
        # Indicadores de Saúde Pública
        round(col("prev_depressao_regiao_pct"), 2).alias("percentual_prevalencia_depressao"),
        round(col("psiquiatras_por_100mil"), 2).alias("densidade_psiquiatras_por_100mil"),
        
        # Criação do KPI de vendas proporcionais por habitante
        round((col("unidades_vendidas_est") / col("populacao_2022")) * 100000, 2)
        .alias("kpi_vendas_por_100mil_hab")
    )
    # Filtro de Linhas: Mantém apenas os dados históricos mais recentes
    .filter(col("ano_referencia") >= 2019)
)

# Salva o resultado final tratado pelo Power BI
(
    tabela_limpa.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.antidepressivos_limpos_bi")
)

# AUDITORIA
print(f"Processamento concluído com sucesso!")
print(f"Quantidade de registros brutos: {dados_brutos.count()}")
print(f"Quantidade de registros limpos: {tabela_limpa.count()}")

# linhas de prévia
display(tabela_limpa.limit(15))

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Dados ──────────────────────────────────────────────────────────────────────
df = spark.table("workspace.default.antidepressivos_limpos_bi").toPandas()

TARGET   = "kpi_vendas_por_100mil_hab"
FEATURES = ["ano_referencia", "percentual_prevalencia_depressao", "densidade_psiquiatras_por_100mil", "regiao_enc"]

le = LabelEncoder()
df["regiao_enc"] = le.fit_transform(df["regiao_nome"].fillna("DESCONHECIDO"))

df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

# ── Treino / Teste ─────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

# ── Modelo ─────────────────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train_sc, y_train)

# ── Avaliação ──────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test_sc)

print(f"MAE  : {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

# ── Gráficos ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Real vs Predito
lim = [min(y_test.min(), y_pred.min()) * 0.95, max(y_test.max(), y_pred.max()) * 1.05]
axes[0].scatter(y_test, y_pred, alpha=0.5, edgecolors="k", linewidths=0.4)
axes[0].plot(lim, lim, "r--", lw=1.5, label="Ideal")
axes[0].set(xlabel="Real", ylabel="Predito", title="Real vs Predito", xlim=lim, ylim=lim)
axes[0].legend()

# Resíduos
residuos = y_test - y_pred
axes[1].hist(residuos, bins=30, edgecolor="k", color="steelblue", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set(xlabel="Resíduo", ylabel="Frequência", title="Distribuição dos Resíduos")

plt.suptitle("Regressão Linear — Antidepressivos Brasil", fontsize=13)
plt.tight_layout()
plt.show()

# Importância (coeficientes padronizados)
imp = (
    pd.DataFrame({"feature": FEATURES, "coef": model.coef_})
    .sort_values("coef", key=abs, ascending=True)
)



In [0]:
%pip install prophet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

# ── Dados ──────────────────────────────────────────────────────────────────────
df_raw = spark.table("workspace.default.antidepressivos_limpos_bi").toPandas()

df_ts = (
    df_raw
    .groupby("ano_referencia", as_index=False)
    .agg(
        y           = ("kpi_vendas_por_100mil_hab",        "mean"),
        prevalencia = ("percentual_prevalencia_depressao", "mean"),
        psiquiatras = ("densidade_psiquiatras_por_100mil",  "mean"),
    )
    .assign(ds=lambda d: pd.to_datetime(d["ano_referencia"].astype(str) + "-01-01"))
    .sort_values("ds")
    .reset_index(drop=True)
)

# ── Modelo ─────────────────────────────────────────────────────────────────────
model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.15,
    interval_width=0.95,
)
model.add_regressor("prevalencia")
model.add_regressor("psiquiatras")
model.fit(df_ts[["ds", "y", "prevalencia", "psiquiatras"]])

# ── Previsão ───────────────────────────────────────────────────────────────────
ANOS_FUTURO = 4

# Usa a média histórica como fallback seguro (evita NaN caso o último valor seja nulo)
fallback_prev = df_ts["prevalencia"].dropna().mean()
fallback_psiq = df_ts["psiquiatras"].dropna().mean()

future = model.make_future_dataframe(periods=ANOS_FUTURO, freq="YS")

# Normaliza as datas para comparação sem timezone
future["ds_norm"] = pd.to_datetime(future["ds"]).dt.normalize()
df_ts["ds_norm"]  = pd.to_datetime(df_ts["ds"]).dt.normalize()

lookup_prev = df_ts.set_index("ds_norm")["prevalencia"].to_dict()
lookup_psiq = df_ts.set_index("ds_norm")["psiquiatras"].to_dict()

future["prevalencia"] = future["ds_norm"].apply(lambda d: lookup_prev.get(d, fallback_prev))
future["psiquiatras"] = future["ds_norm"].apply(lambda d: lookup_psiq.get(d, fallback_psiq))

# Garante que não haja NaN antes de prever
future["prevalencia"] = future["prevalencia"].fillna(fallback_prev)
future["psiquiatras"] = future["psiquiatras"].fillna(fallback_psiq)

future = future.drop(columns=["ds_norm"])
df_ts  = df_ts.drop(columns=["ds_norm"])

forecast = model.predict(future)

# ── Gráficos ───────────────────────────────────────────────────────────────────
model.plot(forecast, figsize=(12, 5))
plt.title("Previsão — KPI Vendas / 100 mil hab.")
plt.tight_layout()
plt.show()

model.plot_components(forecast)
plt.tight_layout()
plt.show()

# Gráfico customizado com IC e separação histórico / futuro
fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"], alpha=0.2, color="royalblue", label="IC 95%")
ax.plot(forecast["ds"], forecast["yhat"], color="royalblue", lw=2, label="Previsão")
ax.scatter(df_ts["ds"], df_ts["y"], color="black", zorder=5, s=60, label="Dados reais")
ax.axvline(df_ts["ds"].max(), color="red", linestyle="--", lw=1.2, label="Início da projeção")
ax.set(title="Série Temporal + Projeção — Prophet", xlabel="Ano", ylabel="KPI Vendas / 100 mil hab.")
ax.legend()
plt.tight_layout()
plt.show()

# ── Validação cruzada ──────────────────────────────────────────────────────────
try:
    df_cv   = cross_validation(model, initial="730 days", period="365 days", horizon="365 days", disable_tqdm=True)
    df_perf = performance_metrics(df_cv)
    display(df_perf[["horizon", "mae", "rmse", "mape"]])
except Exception as e:
    print(f"Validação cruzada ignorada: {e}")

# ── Exporta tabela Delta ───────────────────────────────────────────────────────
forecast_out = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].copy()
forecast_out.columns = ["data_referencia", "previsao", "ic_inferior_95", "ic_superior_95"]

spark.createDataFrame(forecast_out).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.antidepressivos_previsao_prophet")

print("Tabela salva: antidepressivos_previsao_prophet")
